# ML-05 — Feature Vector and Leakage/Privacy Check

Runs **offline** on the bundled anonymized slice (30,000 pages). This notebook builds the honest
feature vector used at the bundled scale, documents every field, and **attacks its own features**
with a deliberate leak to show what leakage looks like in a number.

> The full-release feature set (March 2026 → April 2026) and its leakage audit live in
> `w05_model.ipynb` (features) and `w06_validation_audit.ipynb` (knowability + leak trap). This
> notebook covers the same discipline at the offline scale.

## 1. Build the feature vector

Unit of analysis: **one row = one content page**. Features are numeric page/search signals plus
categorical page descriptors, all taken from the *same 90-day snapshot* as the proxy label.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(root / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

NUM_FEATURES = ["search_volume", "competition", "cpc", "word_count", "char_count",
                "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
                "days_with_impressions", "days_with_sessions", "content_age_days",
                "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
                "ai_traffic_pct"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

X = df[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
X = pd.concat([X, pd.get_dummies(df[CAT_FEATURES].astype(str), dtype=float)], axis=1)
print("feature matrix:", X.shape, "| rows", len(df), "| target base", round(df['is_declining_label'].mean(), 3))

feature matrix: (30000, 52) | rows 30000 | target base 0.542


## 2. Feature notes (meaning, missing, categorical, available-when?)

| feature | meaning | missing-handling | available before the moment we predict? |
|---|---|---|---|
| `log_impressions_90d` | log(1+90-day impressions) | none | yes — trailing snapshot |
| `log_clicks_90d` | log(1+90-day clicks) | none | yes |
| `ctr` | clicks/impressions | 0 kept as 0 | yes |
| `avg_position` | 90-day average rank | `0` = unpositioned, kept | yes |
| `days_with_impressions` | coverage days | none | yes |
| `days_since_last_update` | freshness | none | yes **on this slice**; see §4 — as-of-release on the warehouse |
| `word_count`, `char_count` | depth | 0 = unknown | yes |
| `scroll_rate`, `engagement_rate` | GA4 engagement | sparse; 0 = unknown | yes but sparse |
| categorical tiers | bucketed descriptors | `"unknown"` | yes |

The proxy label (`trend_direction == 'down'`) is **same-window**, so *any column derived from the
window comparison is effectively the label in disguise* — that is exactly what §3 proves.

## 3. The leakage hunt

The attack: add one seductive feature — momentum, `impressions_last_30d / impressions_prev_30d`.
It looks like a normal signal, but the slice's `trend_direction` is itself computed from that same
30d-vs-30d comparison, so it *is* the label. Same split, same model, one column difference.

In [2]:
def cv_auc(frame, y, groups, seed=42):
    from sklearn.model_selection import GroupShuffleSplit
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed).split(frame, y, groups))
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=15, max_features="sqrt",
                                random_state=seed, n_jobs=-1).fit(frame.iloc[tr], y[tr])
    return round(float(roc_auc_score(y[te], rf.predict_proba(frame.iloc[te])[:, 1])), 3)

y = df["is_declining_label"].to_numpy()
groups = df["client_id"].to_numpy()          # grouped split: whole clients held out
honest_auc = cv_auc(X, y, groups)

leaky = X.copy()
leaky["momentum_30d_vs_prev"] = (df["impressions_last_30d"] /
                                 df["impressions_prev_30d"].clip(lower=1)).replace([np.inf, -np.inf], 0).fillna(0)
leak_auc = cv_auc(leaky, y, groups)

# a second, subtler leak: raw trend_pct straight through
leaky2 = X.copy()
leaky2["trend_pct"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)
leak2_auc = cv_auc(leaky2, y, groups)

print(f"honest feature set (grouped-by-client)   AUC = {honest_auc}")
print(f"+ momentum (last30/prev30)               AUC = {leak_auc}   <-- the label in disguise")
print(f"+ raw trend_pct                          AUC = {leak2_auc}")
print()
print("VERDICT: a single label-derived column lifts AUC toward 1.0 on the SAME split. That is")
print("leakage, not skill. Rule applied: on this slice NO feature may be derived from the")
print("30d-vs-30d comparison that defines trend_direction.")

honest feature set (grouped-by-client)   AUC = 0.623
+ momentum (last30/prev30)               AUC = 0.996   <-- the label in disguise
+ raw trend_pct                          AUC = 1.0

VERDICT: a single label-derived column lifts AUC toward 1.0 on the SAME split. That is
leakage, not skill. Rule applied: on this slice NO feature may be derived from the
30d-vs-30d comparison that defines trend_direction.


## 4. What I excluded and why

| excluded column | why |
|---|---|
| `trend_direction`, `trend_pct` | they *define* the proxy label → tautology |
| `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d` | inputs to `trend_pct` → leak the label on this slice |
| `impression_tier`, `position_tier` as raw numerics | derived from impressions/position already in the vector (keep as categorical descriptors instead) |
| `client_id`, `content_id` | identifiers — join/group keys only, **never** features; no client names/URLs/queries are present |
| GA4 engagement as a hard filter | sparse here; used as a feature with 0=unknown, never to drop rows silently |

On the **warehouse release** the same discipline becomes stricter: `content_updated_date` is
excluded because it is an as-of-release snapshot, and the label is a genuine prior-window →
future-window decline (see `w06_validation_audit.ipynb`).

In [3]:
metrics = {
    "task": "ml-05",
    "data": "data/raw/content_refresh_anonymized.csv",
    "rows": int(len(df)),
    "feature_matrix_shape": list(X.shape),
    "target": "is_declining_label = (trend_direction == 'down') [same-window proxy]",
    "honest_auc": honest_auc,
    "leak_momentum_auc": leak_auc,
    "leak_trend_pct_auc": leak2_auc,
    "excluded": ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
                 "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
                 "client_id", "content_id"],
    "decision": "no feature derived from the label's 30d-vs-30d window is allowed",
}
(OUT / "w03_feature_leakage_metrics.json").write_text(json.dumps(metrics, indent=2))
print("metrics receipt written:", OUT / "w03_feature_leakage_metrics.json")

metrics receipt written: C:\Users\Ananya\OneDrive\文件\FlyRank Intern\flyrank-ml-internship-start\work\outputs\w03_feature_leakage_metrics.json


### Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed offline on the bundled slice)
- [x] No client names, URLs, or private queries anywhere — identifiers are join keys only
- [x] The leakage test is shown in numbers (honest vs leaky AUC)
- [x] Committed to my repo under `work/notebooks/`